# 04f（US）Resonance：60 甲子周期的谐波检验（`jiazi_idx`；k=5/6；`ret_1d`）

目的：把 `stem/branch` 热力图中的“条纹直觉”回到 60 周期（`jiazi_idx=0..59`）做可审计检验。

输入：
- `data/clean_us/market_ganzhi_{ts_code}.csv.gz`

输出（写入 `data/clean_us/robustness/`）：
- `resonance_harmonic_k56_{ts_code}_ret_1d.csv` / 汇总 `resonance_harmonic_k56_ret_1d.csv`
- `resonance_after_additive_{ts_code}_ret_1d.csv` / 汇总 `resonance_after_additive_ret_1d.csv`
- `resonance_meta_k56_ret_1d.csv`（跨指数 Fisher 合并）

注意：该检验回答“周期成分是否存在”，不等价于机制/因果证明。


In [1]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns



from matplotlib import font_manager, rcParams


def set_chinese_font() -> None:
    # Best-effort Chinese font setup for Matplotlib.
    # - Tries common Linux CJK fonts
    # - On WSL, also tries registering Windows fonts under /mnt/c/Windows/Fonts

    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK TC',
        'Noto Sans CJK JP',
        'Source Han Sans SC',
        'WenQuanYi Micro Hei',
        'Microsoft YaHei',
        'SimHei',
        'SimSun',
        'Arial Unicode MS',
    ]

    # WSL fallback: register Windows fonts if available
    try:
        from pathlib import Path as _Path

        win_dir = _Path('/mnt/c/Windows/Fonts')
        win_files = [
            win_dir / 'msyh.ttc',
            win_dir / 'msyhbd.ttc',
            win_dir / 'simhei.ttf',
            win_dir / 'simsun.ttc',
            win_dir / 'arialuni.ttf',
        ]

        extra_names: list[str] = []
        addfont = getattr(font_manager.fontManager, 'addfont', None)
        for fp in win_files:
            try:
                if fp.is_file():
                    if callable(addfont):
                        addfont(str(fp))
                    extra_names.append(font_manager.FontProperties(fname=str(fp)).get_name())
            except Exception:
                continue

        candidates = [n for n in extra_names if n] + candidates
    except Exception:
        pass

    for name in candidates:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.sans-serif'] = [name]
            rcParams['axes.unicode_minus'] = False
            print(f'Using Chinese font: {name}')
            return
        except Exception:
            continue

    rcParams['axes.unicode_minus'] = False
    print(
        'Warning: no Chinese font found; Chinese labels may not render. '
        'If you are on WSL, ensure Windows fonts are accessible under /mnt/c/Windows/Fonts; '
        'or install Noto CJK fonts in Linux (e.g., fonts-noto-cjk).'
    )


set_chinese_font()

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean_us'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['spx', 'ndq', 'ndx', 'dji']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks_US 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out




Using Chinese font: Microsoft YaHei
PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支


## 1) 谐波检验 + 诊断（回归掉 stem+branch 后再检）


In [2]:
# Resonance test: harmonic components on jiazi_idx (period=60)
RESONANCE_PERIOD = 60
RESONANCE_KS = [5, 6]  # k=5 -> 12-day cycle, k=6 -> 10-day cycle


def add_harmonic_features(
    df: pd.DataFrame,
    *,
    period: int = RESONANCE_PERIOD,
    ks: list[int] = RESONANCE_KS,
) -> pd.DataFrame:
    out = df.copy()
    if 'jiazi_idx' not in out.columns:
        raise ValueError('Missing jiazi_idx in df')

    j = pd.to_numeric(out['jiazi_idx'], errors='coerce')
    out = out.assign(jiazi_idx=j).dropna(subset=['jiazi_idx']).copy()
    out['jiazi_idx'] = out['jiazi_idx'].astype(int)

    # Keep only valid 0..period-1
    out = out[out['jiazi_idx'].between(0, period - 1)].copy()

    i = out['jiazi_idx'].to_numpy(dtype=float)
    for k in ks:
        ang = 2.0 * np.pi * k * i / float(period)
        out[f's{k}'] = np.sin(ang)
        out[f'c{k}'] = np.cos(ang)
    return out


def harmonic_amp_phase(beta_sin: float, beta_cos: float) -> tuple[float, float]:
    # beta_sin * sin(wt) + beta_cos * cos(wt) = A * sin(wt + phi)
    amp = float(np.sqrt(beta_sin ** 2 + beta_cos ** 2))
    phase = float(np.arctan2(beta_cos, beta_sin))
    return amp, phase


def wald_joint_test(res, terms: list[str]) -> dict:
    hyp = ', '.join([f'{t} = 0' for t in terms])
    try:
        wt = res.wald_test(hyp)
        stat = float(np.asarray(getattr(wt, 'statistic', np.nan)).ravel()[0])
        p = float(np.asarray(getattr(wt, 'pvalue', np.nan)).ravel()[0])
        df_num = int(getattr(wt, 'df_num', len(terms)))
        return {'wald_stat': stat, 'df_num': df_num, 'p_value': p}
    except Exception:
        return {'wald_stat': np.nan, 'df_num': len(terms), 'p_value': np.nan}


def fit_resonance_models(ts_code: str) -> tuple[dict, dict]:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)

    df = df.dropna(subset=['ret_1d', 'jiazi_idx', 'weekday', 'month', 'year', 'stem', 'branch']).copy()
    df = add_harmonic_features(df)

    # Jiazi group sizes (sanity check)
    j_counts = (
        df.groupby('jiazi_idx').size().reindex(range(RESONANCE_PERIOD), fill_value=0).to_numpy()
    )
    n_min_jiazi = int(j_counts.min())
    n_med_jiazi = int(np.median(j_counts))
    n_max_jiazi = int(j_counts.max())
    n_missing_jiazi = int(np.sum(j_counts == 0))


    # Ensure categoricals for controls
    for col in ['weekday', 'month', 'year']:
        df[col] = df[col].astype('category')

    base_formula = 'ret_1d ~ C(weekday) + C(month) + C(year)'
    harm_formula = 'ret_1d ~ s5 + c5 + s6 + c6 + C(weekday) + C(month) + C(year)'

    base_res = smf.ols(base_formula, data=df).fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})
    harm_res = smf.ols(harm_formula, data=df).fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})

    terms = ['s5', 'c5', 's6', 'c6']
    jt = wald_joint_test(harm_res, terms)

    b = {t: float(harm_res.params.get(t, np.nan)) for t in terms}
    amp5, phase5 = harmonic_amp_phase(b['s5'], b['c5'])
    amp6, phase6 = harmonic_amp_phase(b['s6'], b['c6'])

    rec_main = {
        'ts_code': ts_code,
        'n_days': int(len(df)),
        'n_min_jiazi': int(n_min_jiazi),
        'n_med_jiazi': int(n_med_jiazi),
        'n_max_jiazi': int(n_max_jiazi),
        'n_missing_jiazi': int(n_missing_jiazi),
        'min_trade_date': str(df['trade_date'].min()),
        'max_trade_date': str(df['trade_date'].max()),
        'p_wald_k56': float(jt['p_value']),
        'wald_stat_k56': float(jt['wald_stat']),
        'df_num_k56': int(jt['df_num']),
        'beta_s5': b['s5'],
        'beta_c5': b['c5'],
        'beta_s6': b['s6'],
        'beta_c6': b['c6'],
        'amp_5': float(amp5),
        'phase_5_rad': float(phase5),
        'amp_6': float(amp6),
        'phase_6_rad': float(phase6),
        'r2_controls': float(getattr(base_res, 'rsquared', np.nan)),
        'r2_harmonic': float(getattr(harm_res, 'rsquared', np.nan)),
        'delta_r2': float(getattr(harm_res, 'rsquared', np.nan) - getattr(base_res, 'rsquared', np.nan)),
        'hac_maxlags': int(HAC_MAXLAGS),
        'formula': harm_formula,
    }

    # Diagnostic: remove additive main effects (stem + branch) then test harmonics on residuals
    add_formula = 'ret_1d ~ C(stem) + C(branch) + C(weekday) + C(month) + C(year)'
    add_res = smf.ols(add_formula, data=df).fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})

    df2 = df.copy()
    df2['ret_resid_additive'] = add_res.resid

    resid_formula = 'ret_resid_additive ~ s5 + c5 + s6 + c6'
    resid_res = smf.ols(resid_formula, data=df2).fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})

    jt2 = wald_joint_test(resid_res, terms)
    rec_resid = {
        'ts_code': ts_code,
        'n_days': int(len(df2)),
        'n_min_jiazi': int(n_min_jiazi),
        'n_med_jiazi': int(n_med_jiazi),
        'n_max_jiazi': int(n_max_jiazi),
        'n_missing_jiazi': int(n_missing_jiazi),
        'min_trade_date': str(df2['trade_date'].min()),
        'max_trade_date': str(df2['trade_date'].max()),
        'p_wald_k56_resid_additive': float(jt2['p_value']),
        'wald_stat_k56_resid_additive': float(jt2['wald_stat']),
        'df_num_k56': int(jt2['df_num']),
        'hac_maxlags': int(HAC_MAXLAGS),
        'formula_additive': add_formula,
        'formula_resid': resid_formula,
    }

    return rec_main, rec_resid


# Run (per ts_code) + cross-index Fisher combine
res_rows = []
resid_rows = []

for ts_code in TS_CODES:
    try:
        rec_main, rec_resid = fit_resonance_models(ts_code)
    except Exception as exc:
        print(f'[{ts_code}] resonance failed:', exc)
        continue

    res_rows.append(rec_main)
    resid_rows.append(rec_resid)

    p1 = ROBUST_DIR / f'resonance_harmonic_k56_{ts_code}_ret_1d.csv'
    pd.DataFrame([rec_main]).to_csv(p1, index=False)
    print('saved:', p1)

    p2 = ROBUST_DIR / f'resonance_after_additive_{ts_code}_ret_1d.csv'
    pd.DataFrame([rec_resid]).to_csv(p2, index=False)
    print('saved:', p2)


res_df = pd.DataFrame.from_records(res_rows)
if not res_df.empty and 'p_wald_k56' in res_df.columns:
    res_df['q_wald_k56'] = fdr_bh(res_df['p_wald_k56'].to_numpy())
    out_path = ROBUST_DIR / 'resonance_harmonic_k56_ret_1d.csv'
    res_df.to_csv(out_path, index=False)
    print('saved:', out_path)

resid_df = pd.DataFrame.from_records(resid_rows)
if not resid_df.empty and 'p_wald_k56_resid_additive' in resid_df.columns:
    resid_df['q_wald_k56_resid_additive'] = fdr_bh(resid_df['p_wald_k56_resid_additive'].to_numpy())
    out_path2 = ROBUST_DIR / 'resonance_after_additive_ret_1d.csv'
    resid_df.to_csv(out_path2, index=False)
    print('saved:', out_path2)

meta_row = {
    'method': 'fisher',
    'n_indices_used': 0,
    'fisher_stat': np.nan,
    'p_meta_k56': np.nan,
    'hac_maxlags': int(HAC_MAXLAGS),
    'ts_codes_used': '',
}

pvals = []
if not res_df.empty and 'p_wald_k56' in res_df.columns:
    pvals = [float(p) for p in res_df['p_wald_k56'].dropna().tolist()]

if pvals:
    stat_f, p_meta = stats.combine_pvalues(pvals, method='fisher')
    meta_row.update(
        {
            'n_indices_used': int(len(pvals)),
            'fisher_stat': float(stat_f),
            'p_meta_k56': float(p_meta),
            'ts_codes_used': ','.join(res_df['ts_code'].astype(str).tolist()),
            'p_values': ','.join([f'{p:.6g}' for p in pvals]),
        }
    )

meta_path = ROBUST_DIR / 'resonance_meta_k56_ret_1d.csv'
pd.DataFrame([meta_row]).to_csv(meta_path, index=False)
print('saved:', meta_path)

print('\n[resonance] summary (harmonic k=5/6):')
if not res_df.empty:
    view_cols = ['ts_code', 'n_days', 'p_wald_k56', 'q_wald_k56', 'amp_5', 'amp_6', 'delta_r2']
    cols = [c for c in view_cols if c in res_df.columns]
    print(res_df[cols].sort_values('p_wald_k56').to_string(index=False))


saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_harmonic_k56_spx_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_after_additive_spx_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_harmonic_k56_ndq_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_after_additive_ndq_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_harmonic_k56_ndx_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_after_additive_ndx_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_harmonic_k56_dji_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_after_additive_dji_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_harmonic_k56_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\resonance_after_additive_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\